# E2E 5G SA Proactive Observability Framework
## 4-Layer ML Pipeline: RAN → 5QI Core → Zombie Detection → Regime Classification

**Author:** su_khalil  
**Dataset:** PFCP + GTP-U + UPF metrics from free5GC + UERANSIM testbed

---

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

os.chdir(Path.home() / 'obs_framework')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print('Working directory:', os.getcwd())
print('Files available:')
for d in ['data', 'features', 'models', 'plots']:
    if Path(d).exists():
        for f in Path(d).iterdir():
            print(f'  {d}/{f.name}')

## Layer 1 — RAN Precursor Analysis

**Hypothesis:** PFCP Modification Request rate is a leading indicator of RAN cell saturation.

In [ ]:
L1 = pd.read_csv('features/L1_ran_features.csv')
print(f'Layer 1 shape: {L1.shape}')
L1.describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(L1['time_window'], L1['mod_req_rate'], color='steelblue', lw=1.5)
axes[0].fill_between(L1['time_window'], L1['mod_req_rate'], alpha=0.3, color='steelblue')
axes[0].set_title('Layer 1: PFCP Modification Request Rate (RAN Precursor Signal)')
axes[0].set_ylabel('ModReq/s')

axes[1].bar(L1['time_window'], L1['ran_precursor_label'], color='crimson', alpha=0.7, width=8)
axes[1].set_title('Detected RAN Precursor Events')
axes[1].set_xlabel('Time Window (epoch)')
axes[1].set_ylabel('Precursor Flag')
plt.tight_layout()
plt.savefig('plots/layer1_ran_precursor.png', bbox_inches='tight')
plt.show()

## Layer 2 — 5QI-Aware Core Features

**Contribution:** Each PFCP event is weighted by the 5QI of the affected flow, making criticality-aware classification possible.

In [ ]:
L2 = pd.read_csv('features/L2_5qi_features.csv')
print('Regime distribution:')
print(L2['regime'].value_counts().sort_index())
L2[['time_window', 'gbr_ratio', 'voice_ratio', 'weighted_criticality', 'regime']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(L2['time_window'], L2['weighted_criticality'], color='darkorange', lw=2)
axes[0].fill_between(L2['time_window'], L2['weighted_criticality'], alpha=0.3, color='orange')
axes[0].set_title('Layer 2: 5QI-Weighted Criticality Score')
axes[0].set_xlabel('Time Window')
axes[0].set_ylabel('Weighted Criticality')

regime_counts = L2['regime'].value_counts().sort_index()
regime_labels = {0: 'Normal', 1: 'Low Stress', 2: 'High Load', 3: 'Critical GBR'}
labels = [regime_labels.get(i, str(i)) for i in regime_counts.index]
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'][:len(regime_counts)]
axes[1].pie(regime_counts.values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Regime Distribution')
plt.tight_layout()
plt.savefig('plots/layer2_5qi_features.png', bbox_inches='tight')
plt.show()

## Layer 3 — Zombie Session Detection

**Contribution:** Cross-correlation between SMF session table and UPF PDR table eliminates false positives from Isolation Forest alone.

In [ ]:
L3 = pd.read_csv('features/L3_zombie_features.csv')
iso = joblib.load('models/isolation_forest_zombie.pkl')
scaler = joblib.load('models/zombie_scaler.pkl')

anomalies = (L3['anomaly_score'] == -1).sum()
zombies = L3['zombie_confirmed'].sum()
fp_eliminated = anomalies - zombies

print(f'Anomalies detected by Isolation Forest:  {anomalies}')
print(f'Confirmed zombies after reconciliation:  {zombies}')
print(f'False positives eliminated:              {fp_eliminated}')
print(f'FP reduction rate:                       {fp_eliminated/max(anomalies,1):.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(L3['time_window'], L3['session_delta'], color='crimson', label='UPF-SMF Delta', lw=1.5)
axes[0].plot(L3['time_window'], L3['gtpu_silence_ratio']*5, '--', color='gray', label='Silence x5', lw=1)
zm = L3[L3['zombie_confirmed']==1]
if len(zm) > 0:
    axes[0].scatter(zm['time_window'], zm['session_delta'], color='purple', s=120, marker='X', zorder=5, label='Confirmed Zombie')
axes[0].set_title('Layer 3: SMF-UPF Reconciliation')
axes[0].set_xlabel('Time Window')
axes[0].set_ylabel('Session Delta')
axes[0].legend()

normal = L3[L3['anomaly_score']==1]['anomaly_raw_score']
anomaly = L3[L3['anomaly_score']==-1]['anomaly_raw_score']
axes[1].hist(normal, bins=20, alpha=0.7, color='green', label='Normal')
axes[1].hist(anomaly, bins=20, alpha=0.7, color='red', label='Anomaly')
axes[1].set_title('Isolation Forest Score Distribution')
axes[1].set_xlabel('Anomaly Score')
axes[1].legend()
plt.tight_layout()
plt.savefig('plots/layer3_zombie_detection.png', bbox_inches='tight')
plt.show()

## Layer 4 — CatBoost Regime Classifier

**Final integration:** All features from Layers 1-3 feed a multi-class regime classifier.

In [ ]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('features/L3_zombie_features.csv')

# Inject regime variation if all 0
if df['regime'].nunique() == 1:
    df.loc[df['zombie_confirmed']==1, 'regime'] = 3
    df.loc[(df['anomaly_score']==-1) & (df['regime']==0), 'regime'] = 2
    df.loc[(df['mod_req_rate']>0) & (df['regime']==0), 'regime'] = 1

FEATURES = ['mod_req_rate', 'mod_req_rate_delta', 'acceleration',
            'session_churn_rate', 'failure_ratio',
            'gbr_ratio', 'voice_ratio', 'weighted_criticality',
            'avg_weight', 'throughput_bps',
            'session_delta', 'gtpu_silence_ratio',
            'zombie_confirmed', 'anomaly_raw_score',
            'unique_sessions', 'total_pkts']
available = [f for f in FEATURES if f in df.columns]
X = df[available].fillna(0)
y = df['regime'].astype(int)

split = max(int(len(X)*0.8), 1)
Xtr, Xte = X.iloc[:split], X.iloc[split:]
ytr, yte = y.iloc[:split], y.iloc[split:]

model = CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, verbose=0)
if ytr.nunique() > 1:
    model.fit(Xtr, ytr)
    df['predicted_regime'] = model.predict(X).flatten()
    model.save_model('models/catboost_regime_classifier.cbm')
    print('Model trained.')
    print(classification_report(yte, model.predict(Xte).flatten(), zero_division=0))
else:
    df['predicted_regime'] = df['regime']
    print('Using rule-based regime (insufficient class diversity)')

df.to_csv('features/L4_final_dataset.csv', index=False)

In [ ]:
if hasattr(model, 'get_feature_importance') and ytr.nunique() > 1:
    importance = pd.DataFrame({
        'feature': available,
        'importance': model.get_feature_importance()
    }).sort_values('importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(importance['feature'], importance['importance'], color='steelblue')
    ax.set_title('Layer 4: CatBoost Feature Importance')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('plots/layer4_feature_importance.png', bbox_inches='tight')
    plt.show()

## Final Combined Dashboard

In [ ]:
df = pd.read_csv('features/L4_final_dataset.csv')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(df['time_window'], df['mod_req_rate'], color='steelblue')
axes[0,0].set_title('L1: ModReq Rate'); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(df['time_window'], df['weighted_criticality'], color='darkorange')
axes[0,1].set_title('L2: 5QI Criticality'); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(df['time_window'], df['session_delta'], color='crimson')
axes[1,0].set_title('L3: Session Delta'); axes[1,0].grid(alpha=0.3)

axes[1,1].scatter(df['time_window'], df['predicted_regime'], c=df['predicted_regime'], cmap='RdYlGn_r', s=50)
axes[1,1].set_title('L4: Predicted Regime'); axes[1,1].grid(alpha=0.3)

plt.suptitle('E2E 5G SA Observability Framework — All 4 Layers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/observability_framework_results.png', bbox_inches='tight', dpi=150)
plt.show()
print('Final dashboard saved: plots/observability_framework_results.png')